# Visualising whitened examples (strain + witness)

This notebook plots one **background**, one **signal** and one **glitch** example from the
generated toy dataset, in both the **strain** and the **witness** channel.

For each example we show:
1. the **time series** (time domain), and
2. the **amplitude spectral density** (frequency domain),

using [`gwpy`](https://gwpy.github.io/).

The `data` stored in `background.h5` / `signal.h5` / `glitch.h5` is **already whitened**
(channel order `[strain, witness]`, see the README *Output format* section), so here we simply
wrap each channel into a `gwpy.timeseries.TimeSeries` and plot it — no extra whitening is applied.

> **Prerequisite:** the per-class files must exist. If you have not generated them yet, run
> (from the `dataset/` folder):
> ```bash
> python load_data.py --config configs/config_H1.yaml --data ./data     # download background
> python main.py      --config configs/config_H1.yaml --data ./data/background_data --out ./out
> ```

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from gwpy.timeseries import TimeSeries

In [ ]:
# --- configuration ----------------------------------------------------------
# Folder with the generated per-class HDF5 files (output of dataset/main.py).
OUT_DIR = Path("../dataset/out")

# Sampling rate of the stored data (configs/config_H1.yaml -> general.sample_rate).
SAMPLE_RATE = 4096  # Hz

# Which example index to pull from each file (any value in [0, num_per_class)).
EXAMPLE_INDEX = 0

CLASSES = ["background", "signal", "glitch"]
CHANNELS = ["strain", "witness"]
COLORS = {"strain": "#1f77b4", "witness": "#d62728"}

In [ ]:
def load_example(cls, index=EXAMPLE_INDEX, out_dir=OUT_DIR, sample_rate=SAMPLE_RATE):
    """Load one example of a given class and return {channel_name: TimeSeries}.

    The on-disk ``data`` has shape (N, 2, T) with channel order [strain, witness]
    and is already whitened.
    """
    path = Path(out_dir) / f"{cls}.h5"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found — generate the dataset first (see the note at the top "
            "of this notebook), or point OUT_DIR at the folder holding the .h5 files."
        )
    with h5py.File(path, "r") as f:
        sample = f["data"][index]  # (2, T)
        chan_names = [
            c.decode() if isinstance(c, (bytes, bytearray)) else str(c)
            for c in f.attrs["channels"]
        ]
    return {
        name: TimeSeries(
            sample[i].astype(float),
            sample_rate=sample_rate,
            name=f"{cls} — {name}",
        )
        for i, name in enumerate(chan_names)
    }


examples = {cls: load_example(cls) for cls in CLASSES}
print({cls: {ch: ts.shape for ch, ts in chans.items()} for cls, chans in examples.items()})

## 1. Time domain

Rows are the three classes, columns are the two channels. The witness is *blind* to the
astrophysical signal (signal row, witness column should look like background), but carries a
coupled copy of the glitch (glitch row, witness column).

In [ ]:
fig, axes = plt.subplots(
    len(CLASSES), len(CHANNELS), figsize=(12, 9), sharex=True, sharey="row"
)

for r, cls in enumerate(CLASSES):
    for c, ch in enumerate(CHANNELS):
        ts = examples[cls][ch]
        ax = axes[r, c]
        ax.plot(ts.times.value, ts.value, lw=0.8, color=COLORS[ch])
        ax.set_title(f"{cls} — {ch}")
        ax.grid(True, alpha=0.3)
        if r == len(CLASSES) - 1:
            ax.set_xlabel("Time [s]")
        if c == 0:
            ax.set_ylabel("Whitened amplitude [σ]")

fig.suptitle("Whitened time series", fontsize=14)
fig.tight_layout()
plt.show()

## 2. Frequency domain

Amplitude spectral density (ASD) of the same examples, computed with `gwpy`'s
`TimeSeries.asd`. Because the data is whitened, the background ASD is approximately flat;
signal/glitch power shows up as excess over that flat floor.

In [ ]:
# Short segments (1 s by default) -> use a sub-second FFT length so we still average a few
# segments for a smoother estimate.
duration = examples[CLASSES[0]][CHANNELS[0]].duration.value
fftlength = min(0.5, duration)
overlap = fftlength / 2

fig, axes = plt.subplots(
    len(CLASSES), len(CHANNELS), figsize=(12, 9), sharex=True, sharey=True
)

for r, cls in enumerate(CLASSES):
    for c, ch in enumerate(CHANNELS):
        ts = examples[cls][ch]
        asd = ts.asd(fftlength=fftlength, overlap=overlap)
        ax = axes[r, c]
        ax.loglog(asd.frequencies.value, asd.value, color=COLORS[ch])
        ax.set_title(f"{cls} — {ch}")
        ax.grid(True, which="both", alpha=0.3)
        if r == len(CLASSES) - 1:
            ax.set_xlabel("Frequency [Hz]")
        if c == 0:
            ax.set_ylabel("ASD [1/√Hz]")

fig.suptitle("Whitened amplitude spectral density", fontsize=14)
fig.tight_layout()
plt.show()